# 04 · Policy-Aware Question Answering

**Rules:** answer **only** from retrieved policy evidence, quote it **verbatim**, cite the **`policy_doc_id`**, and refuse instead of guessing when the policies do not cover the question.

In [ ]:
# --- setup: make the project package importable and show the run mode ---
import sys, os, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src                                   # sets macOS OpenMP compatibility flags
from src import config
from src.llm import mode_label
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", palette="Set2"); plt.rcParams["figure.dpi"] = 110
OUT = config.OUTPUTS
print("Mode:", mode_label(), "| outputs ->", OUT.relative_to(ROOT))

In [ ]:
from src import documents
from src.retrieval import policy_index
from src.policy_qa import answer_policy, validate, SYSTEM, REFUSAL
full, chunks = documents.build_policy_docs()
pd.DataFrame(full)

In [ ]:
pd.DataFrame(chunks)[["chunk_id", "title", "text"]]

The 5 policies are short (2 sentences each). Each one is indexed as **sentence-level chunks**, so retrieval can point to the exact rule. The LLM then receives the **full text** of every policy that had a relevant chunk, because the second sentence often holds the condition, for example the "30 percent" rule in POL_RETURNS.

## 1. Retrieval inspection on labelled policy questions

In [ ]:
labelled = [("Can I return shoes after 10 days?", "POL_RETURNS"), ("Is a 40% discounted item returnable?", "POL_RETURNS"),
            ("Can I exchange a shirt for a bigger size?", "POL_EXCHANGE"), ("Can I exchange innerwear?", "POL_EXCHANGE"),
            ("How long does standard delivery take?", "POL_SHIPPING"), ("Is express shipping available?", "POL_SHIPPING"),
            ("When will I get my money back?", "POL_REFUND"), ("Do refunds need a quality check?", "POL_REFUND"),
            ("Does my watch have a warranty?", "POL_WARRANTY"), ("Is a cracked watch glass covered?", "POL_WARRANTY")]
idx = policy_index(); rows = []
for q, gold in labelled:
    hits = idx.search(q, k=3)
    rows.append({"question": q, "gold": gold, "top1": hits[0]["doc_id"], "top1_score": hits[0]["score"],
                 "hit@1": hits[0]["doc_id"] == gold, "hit@3": gold in [h["doc_id"] for h in hits]})
pol_eval = pd.DataFrame(rows); pol_eval.to_csv(OUT / "04_policy_retrieval.csv", index=False)
print(f"policy retrieval hit@1 = {pol_eval['hit@1'].mean():.0%}, hit@3 = {pol_eval['hit@3'].mean():.0%}")
pol_eval

## 2. Grounded answers with citations and quotes

Prompt:

In [ ]:
print(SYSTEM)

In [ ]:
def show(r):
    print("Q:", r["question"]); print("trace:"); [print("   ", t) for t in r["trace"]]
    print("ANSWER:", r["answer"]); print("citations:", r["citations"], "| quotes:", r["quotes"], "| covered:", r.get("covered"))
    print("=" * 110)
for q in ["Can I return a discounted product after 7 days?", "I bought a jacket at 40% off. Can I return it within 5 days if unused?",
          "Can I exchange a T-shirt for a different size after 8 days?", "My watch strap broke after 2 months - is it covered?",
          "When will I get my refund for a prepaid order?"]:
    show(answer_policy(q))

The policy answers are **conservative**:
- They start with yes, no or "it depends", followed by the exact policy sentence and its `doc_id`.
- When the evidence only partly covers the question, the answer says what the policy *does* state, marks `covered = false`, and suggests contacting support. For example, the warranty policy does not say whether a broken strap is a defect or damage.

## 3. Questions the policies do not cover → refusal, no citation

In [ ]:
for q in ["Do you price-match other websites?", "Can I pay with cryptocurrency?", "What is your loyalty points expiry?"]:
    show(answer_policy(q))

## 4. The citation validator (what stops a hallucinated rule)

In [ ]:
ev = [{"doc_id": "POL_RETURNS", "full_text": documents.load_policies()["POL_RETURNS"] if hasattr(documents, "load_policies") else full[0]["text"]}]
bad = {"answer": "Yes, returns are accepted within 30 days.", "citations": ["POL_RETURNS"], "quotes": ["Returns are allowed within 30 days of delivery"]}
wrong_doc = {"answer": "...", "citations": ["POL_GIFTCARD"], "quotes": []}
good = {"answer": "...", "citations": ["POL_RETURNS"], "quotes": ["Returns are allowed within 7 days of delivery for unused products with original packaging."]}
for name, a in [("invented quote", bad), ("unretrieved doc id", wrong_doc), ("correct", good)]:
    print(f"{name:20} -> issues: {validate(a, ev) or 'none'}")

Any LLM answer that fails validation is replaced by a purely **extractive** answer: the best-matching policy sentence, quoted with its doc_id. A made-up rule therefore never reaches the customer.